007 DATA CLEANING

- Start with 006_data_only_pedestrians parquet file
- End with 007_data_only_pedestrians parquet file

OBJECTIVES: 
- detect and delete rows with a lot of missing features
- clean 'Sesso' column (A)
- clean 'FondoStradale' column (B)
- clean 'Segnaletica' column (C)
- clean 'particolaritastrade' column (D)
- clean 'Pavimentazione' column (E)
- clean 'NaturaIncidente' column (F)

In [150]:
import pandas as pd
import numpy as np
import pyarrow

In [151]:
df = pd.read_parquet('006_data_only_pedestrians.parquet').copy()
df.shape

(19687, 49)

The dataset currently has 49 columns and 19,687 rows of data. Each row now refers to a pedestrian but contains data about the vehicle involved, the driver and passengers. 

In [152]:
df.columns

Index(['Protocollo', 'TipoVeicolo', 'StatoVeicolo', 'NaturaIncidente',
       'NUM_FERITI', 'NUM_RISERVATA', 'NUM_MORTI', 'NUM_ILLESI', 'Sesso',
       'DataOraIncidente', 'CondizioneAtmosferica', 'Traffico', 'Gruppo',
       'Localizzazione1', 'STRADA1', 'Localizzazione2', 'STRADA2', 'Strada02',
       'Chilometrica', 'DaSpecificare', 'Latitude', 'Longitude',
       'particolaritastrade', 'TipoStrada', 'FondoStradale', 'Pavimentazione',
       'Segnaletica', 'Visibilita', 'Illuminazione', 'Tipolesione', 'Deceduto',
       'DecedutoDopo', 'parked_vehicle_involved', 'driver_injury',
       'driver_deceduto', 'driver_deceduto_dopo', 'driver_Sesso',
       'passenger_no_of_females', 'passenger_no_of_males',
       'passenger_no_of_unknown_sex', 'passengers_killed',
       'passengers_uninjured', 'passengers_injured', 'phase_of_day', 'YEAR',
       'MONTH', 'DATE', 'TIME', 'DAY'],
      dtype='object')

In [153]:
def analyse_column(column_name):
    '''
    Analyzes value counts, percentages, and unique protocols for a column in the DataFrame.
    Also permanently converts empty strings to NaN in the original DataFrame.

    Args:
        column_name (str): The name of the column to analyze.

    Returns:
        pandas.DataFrame: A DataFrame with 'Count', '% per category', and 
                         'Unique Accidents' for each unique value.
    '''

    # Validate input
    if column_name not in df.columns:
        raise ValueError(f"Column '{column_name}' not found in DataFrame")

    # Count empty strings before conversion
    empty_string_count = (df[column_name] == '').sum()

    # Permanently replace empty strings with NaN in the original DataFrame
    df[column_name] = df[column_name].replace('', pd.NA)

    if empty_string_count > 0:
        print(
            f"Converted {empty_string_count} empty strings to NaN in column '{column_name}'")

    # Use the cleaned column from the DataFrame
    cleaned_column = df[column_name]

    # Calculate counts and percentages
    counts = cleaned_column.value_counts(dropna=False)
    percentages = (cleaned_column.value_counts(
        dropna=False, normalize=True) * 100).round(2)

    # Calculate unique protocols for each category
    unique_protocols = []
    for category in counts.index:
        if pd.isna(category):
            # Handle NaN values
            mask = cleaned_column.isna()
        else:
            # Handle non-NaN values
            mask = cleaned_column == category

        protocol_count = df.loc[mask, 'Protocollo'].nunique()
        unique_protocols.append(protocol_count)

    # Create result DataFrame
    result = pd.DataFrame({
        'Count': counts,
        '% per category': percentages,
        'Unique Accidents': unique_protocols
    }, index=counts.index)

    return result

DATA CLEANING: ISOLATING AND DELETING ROWS WHERE A LOT OF INFORMATION IS MISSING

In [154]:
# Missing information is '' or null values
# First we count how many missing values are in each column
missing_counts = ((df.isnull()) | (df == '')).sum()

# Now we identify which columns have missing values, that is the count is greater than zero
columns_with_issues = missing_counts[missing_counts > 0]

# View those columns
print(columns_with_issues)

Sesso                        9
CondizioneAtmosferica      202
Traffico                   220
Localizzazione1            200
STRADA2                  14461
Strada02                  4019
Chilometrica              9245
DaSpecificare            15668
Latitude                  2269
Longitude                 2271
particolaritastrade        202
TipoStrada                 205
FondoStradale              202
Pavimentazione             202
Segnaletica                202
Visibilita                 203
Illuminazione             4389
DecedutoDopo              2048
driver_injury             1004
driver_deceduto           1004
driver_deceduto_dopo     18063
dtype: Int64


There seem to be around 200 missing values in many columns. Are the values missing in different, random protocols, or are there specific protocols missing many features? 

When there are missing values in STRADA2, Strada02, Chilometrica and DaSpecificare, it means that we only have the name of the one road as the location of the accident. However, we are more interested in the date, time and conditions of each accident.

Let's check if the other factors like traffic density, type of road, road surface, etc are missing for specific rows:

In [155]:
def find_rows_with_missing_columns(df, columns_to_check, exact_missing_count):
    '''
    Find rows with exactly the specified number of missing columns.

    Parameters:
    - df: DataFrame to analyze
    - columns_to_check: List of column names to check
    - exact_missing_count: Exact number of columns that must be missing

    Returns:
    - Indices of problematic rows
    '''
    missing_mask = (df[columns_to_check].isnull()) | (
        df[columns_to_check] == '')
    missing_per_row = missing_mask.sum(axis=1)
    problematic_rows = missing_per_row[missing_per_row == exact_missing_count]

    print(f'Found {len(problematic_rows)} rows where exactly {exact_missing_count} out of {len(columns_to_check)} columns are missing')

    return problematic_rows.index.tolist()

In [156]:
columns_to_check = ['Sesso', 'Traffico', 'Localizzazione1', 'particolaritastrade', 'TipoStrada', 'FondoStradale',
                    'Pavimentazione', 'Segnaletica', 'Visibilita', 'Illuminazione']

In [157]:
for col in columns_to_check:
    result = analyse_column(col)
    print(result, '\n')

Converted 9 empty strings to NaN in column 'Sesso'
       Count  % per category  Unique Accidents
Sesso                                         
F      10171           51.66              9672
M       9507           48.29              9199
<NA>       9            0.05                 9 

Converted 220 empty strings to NaN in column 'Traffico'
          Count  % per category  Unique Accidents
Traffico                                         
Normale   12987           65.97             12015
Scarso     4624           23.49              4255
Intenso    1856            9.43              1729
<NA>        220            1.12               211 

Converted 200 empty strings to NaN in column 'Localizzazione1'
                             Count  % per category  Unique Accidents
Localizzazione1                                                     
Strada Urbana                18693           94.95             17281
Statale entro l'abitato        243            1.23               225
Provinciale ent

In [158]:
for i in range(0, 11):
    missing = find_rows_with_missing_columns(df, columns_to_check, i)

Found 15276 rows where exactly 0 out of 10 columns are missing
Found 4196 rows where exactly 1 out of 10 columns are missing
Found 12 rows where exactly 2 out of 10 columns are missing
Found 2 rows where exactly 3 out of 10 columns are missing
Found 0 rows where exactly 4 out of 10 columns are missing
Found 0 rows where exactly 5 out of 10 columns are missing
Found 0 rows where exactly 6 out of 10 columns are missing
Found 0 rows where exactly 7 out of 10 columns are missing
Found 1 rows where exactly 8 out of 10 columns are missing
Found 200 rows where exactly 9 out of 10 columns are missing
Found 0 rows where exactly 10 out of 10 columns are missing


There are indeed 200 rows where there are missing values in 9 columns, as suspected.
There is 1 row where there are missing values in 8 columns. 

We will delete these 201 rows:

In [159]:
print(f'Dataframe shape before: {df.shape}\n')
for i in range(8, 10):
    missing = find_rows_with_missing_columns(df, columns_to_check, i)
    df = df.drop(missing).reset_index(drop=True)
    print(f'Shape after deletion: {df.shape}\n')

Dataframe shape before: (19687, 49)

Found 1 rows where exactly 8 out of 10 columns are missing
Shape after deletion: (19686, 49)

Found 200 rows where exactly 9 out of 10 columns are missing
Shape after deletion: (19486, 49)



## CATEGORIES WITH NO ORDER

Now we can clean the categorical columns with no inherent order:
- (A) 'Sesso' (sex of pedestrian)
- (B) 'FondoStradale' (road conditions)
- (C) 'Segnaletica' (road markings)
- (D) 'particolaritastrade' (road features)
- (E) 'Pavimentazione' (road surface)
- (F) 'NaturaIncidente' (type of accident)

### (A) SESSO - pedestrian

In [160]:
analyse_column('Sesso')

,Count,% per category,Unique Accidents
Sesso,,,
F,10094,51.80,9597
M,9383,48.15,9077
<NA>,9,0.05,9


In [161]:
df['pedestrian_gender'] = df['Sesso'].fillna('U')
df['pedestrian_gender'].value_counts(dropna=False)

pedestrian_gender
F    10094
M     9383
U        9
Name: count, dtype: int64

In [162]:
df.drop(columns='Sesso', inplace=True)

#### (A-ii) SESSO - driver

In [163]:
analyse_column('driver_Sesso')

,Count,% per category,Unique Accidents
driver_Sesso,,,
M,14381,73.80,13252
F,4120,21.14,3837
U,985,5.05,928


In [164]:
df = df.rename(columns={'driver_Sesso': 'driver_gender'})

In [165]:
df['driver_gender'].value_counts(dropna=False)

driver_gender
M    14381
F     4120
U      985
Name: count, dtype: int64

In [166]:
for col in ['driver_gender', 'pedestrian_gender']:
    df[col] = pd.Categorical(
        df[col],
        categories=['M', 'F', 'U'],  # explicit allowed values
        ordered=False
    )

#### (A) SESSO - pedestrian & driver - CLEANING COMPLETE
NEW CATEGORICAL COLUMNS: 
- pedestrian_gender
- driver_gender

### (B) FONDO STRADALE - road conditions

In [167]:
analyse_column('FondoStradale')

,Count,% per category,Unique Accidents
FondoStradale,,,
Asciutto,16179,83.03,14945
Bagnato (pioggia),2704,13.88,2501
Bagnato (umiditÃ in atto),572,2.94,540
Bagnato (brina),10,0.05,10
Sdrucciolevole (pietrisco),6,0.03,6
Viscido da liquidi oleosi,6,0.03,6
Sdrucciolevole (fango),4,0.02,4
Ghiacciato,2,0.01,2
Con neve,1,0.01,1


We will lump together: 
Bagnato (umiditÃ  in atto) with Bagnato (pioggia)

In [168]:
df['FondoStradale'] = df['FondoStradale'].apply(
    lambda x: 'Bagnato (pioggia)' if isinstance(
        x, str) and 'in atto' in x else x
)

Now we lump together the other categories: dry, wet, icy, slippery

In [169]:
# Define the mapping dictionary
road_surface_mapping = {
    'Asciutto': 'dry',
    'Bagnato (pioggia)': 'wet',
    'Bagnato (brina)': 'icy',
    'Con neve': 'icy',
    'Ghiacciato': 'icy',
    'Sdrucciolevole (fango)': 'slippery',
    'Sdrucciolevole (pietrisco)': 'slippery',
    'Sdrucciolevole (terriccio)': 'slippery',
    'Viscido da liquidi oleosi': 'slippery',
    '': np.nan
}

# Apply the mapping
df['FondoStradale'] = df['FondoStradale'].map(
    road_surface_mapping).fillna(df['FondoStradale'])

In [170]:
analyse_column('FondoStradale')

,Count,% per category,Unique Accidents
FondoStradale,,,
dry,16179,83.03,14945
wet,3276,16.81,3041
slippery,17,0.09,17
icy,13,0.07,13
<NA>,1,0.01,1


In [171]:
df['road_conditions'] = df['FondoStradale'].fillna('road_conditions_unknown')
df['road_conditions'].value_counts(dropna=False)

road_conditions
dry                        16179
wet                         3276
slippery                      17
icy                           13
road_conditions_unknown        1
Name: count, dtype: int64

In [172]:
df.drop(columns='FondoStradale', inplace=True)

In [173]:
df["road_conditions"] = pd.Categorical(
    df["road_conditions"],
    categories=["dry", "wet", "slippery", "icy", "road_conditions_unknown"],
    ordered=False
)

#### (B) FONDO STRADALE - road conditions - CLEANING COMPLETE
NEW CATEGORICAL COLUMN: 
- road_conditions

### (C) SEGNALETICA - ROAD MARKINGS

In [174]:
analyse_column('Segnaletica')

,Count,% per category,Unique Accidents
Segnaletica,,,
Verticale ed orizzontale,14327,73.52,13212
Orizzontale,2964,15.21,2745
Assente,1366,7.01,1286
Verticale,771,3.96,720
Temporanea di cantiere,57,0.29,53
<NA>,1,0.01,1


As these categories overlap (road markings on the ground are present in the category 'Orizzontale' and 'Verticale ed orizzontale'), instead of making a single categorical column, we will make four binary columns, where the category 'Verticale ed orizzontale' will have a 1 for both onroad and signposts categories. 

In [175]:
# Create the new columns with default values of 0
df['road_markings_absent'] = 0
df['road_markings_onroad'] = 0
df['road_markings_signposts'] = 0
df['road_markings_temporary'] = 0

# Set values based on 'Segnaletica' column
df.loc[df['Segnaletica'] == 'Assente', 'road_markings_absent'] = 1
df.loc[df['Segnaletica'] == 'Orizzontale', 'road_markings_onroad'] = 1
df.loc[df['Segnaletica'] == 'Temporanea di cantiere',
       'road_markings_temporary'] = 1
df.loc[df['Segnaletica'] == 'Verticale', 'road_markings_signposts'] = 1
df.loc[df['Segnaletica'] == 'Verticale ed orizzontale', [
    'road_markings_onroad', 'road_markings_signposts']] = 1

print('New road markings columns:')
print('road_markings_absent:', df['road_markings_absent'].sum())
print('road_markings_onroad:', df['road_markings_onroad'].sum())
print('road_markings_signposts:', df['road_markings_signposts'].sum())
print('road_markings_temporary:', df['road_markings_temporary'].sum())

New road markings columns:
road_markings_absent: 1366
road_markings_onroad: 17291
road_markings_signposts: 15098
road_markings_temporary: 57


In [176]:
df.drop(columns='Segnaletica', inplace=True)

#### (C) SEGNALETICA - road markings - CLEANING COMPLETE
NEW BINARY COLUMNS:
- road_markings_absent
- road_markings_onroad
- road_markings_signposts
- road_markings_temporary

### (D) PARTICOLARITASTRADE - road features

In [177]:
analyse_column('particolaritastrade')

,Count,% per category,Unique Accidents
particolaritastrade,,,
Rettilineo,13009,66.76,12071
Incrocio,2629,13.49,2416
Intersezione semaforizzata,1463,7.51,1362
Intersezione stradale segnalata,747,3.83,672
Curva a visuale libera,673,3.45,622
Rotatoria,338,1.73,301
Intersezione non regolata/non segnalata,233,1.20,210
Pendenza,182,0.93,165
Curva senza visuale libera,148,0.76,136


We will keep the most populated categories, then lump the others together as 'other':

In [178]:
# Define mapping
road_features_map = {
    'Rettilineo': 'Straight road',
    'Incrocio': 'Intersection',
    'Intersezione semaforizzata': 'Intersection with traffic lights',
    'Intersezione stradale segnalata': 'Intersection with signs',
    'Curva a visuale libera': 'Curve with clear view',
    'Rotatoria': 'Roundabout',
    'Intersezione non regolata/non segnalata': 'Intersection unmarked',
    'Pendenza': 'Slope',
    'Curva senza visuale libera': 'Curve without clear view'
}

# Apply mapping and replace all others with 'Unknown'
df['road_features'] = df['particolaritastrade'].map(
    road_features_map).fillna('road_feature_other')

In [179]:
analyse_column('road_features')

,Count,% per category,Unique Accidents
road_features,,,
Straight road,13009,66.76,12071
Intersection,2629,13.49,2416
Intersection with traffic lights,1463,7.51,1362
Intersection with signs,747,3.83,672
Curve with clear view,673,3.45,622
Roundabout,338,1.73,301
Intersection unmarked,233,1.20,210
Slope,182,0.93,165
Curve without clear view,148,0.76,136


In [180]:
df["road_features"] = pd.Categorical(
    df["road_features"],
    categories=[
        "Straight road",
        "Intersection",
        "Intersection with traffic lights",
        "Intersection with signs",
        "Curve with clear view",
        "Roundabout",
        "Intersection unmarked",
        "Slope",
        "Curve without clear view",
        "road_feature_other"
    ],
    ordered=False
)

In [181]:
df.drop(columns='particolaritastrade', inplace=True)

This done, let's check whether these values are consistent with the road_markings columns. It may be more efficient to collapse all of the intersection categories together. 

In [182]:
intersection_types = [
    "Intersection",
    "Intersection with traffic lights",
    "Intersection with signs",
    "Intersection unmarked"
]

# Step 1: marking counts table
marking_summary = (
    df[df["road_features"].isin(intersection_types)]
    .groupby("road_features", observed=True)[[
        "road_markings_absent",
        "road_markings_onroad",
        "road_markings_signposts",
        "road_markings_temporary"
    ]]
    .sum()
    .astype(int)
    .reindex(intersection_types)   # keep only these 4, in this order
    .rename(columns={
        "road_markings_absent": "Absent",
        "road_markings_onroad": "On road",
        "road_markings_signposts": "Signposts",
        "road_markings_temporary": "Temporary"
    })
)

# Step 2: true totals per road_feature (count of rows)
true_totals = df["road_features"].value_counts().reindex(
    intersection_types).fillna(0).astype(int)

# Step 3: add as a new column
marking_summary["Total rows"] = true_totals.values

print(marking_summary)

                                  Absent  On road  Signposts  Temporary  \
road_features                                                             
Intersection                         129     2350       2185          5   
Intersection with traffic lights      12     1426       1413          5   
Intersection with signs               21      686        666          2   
Intersection unmarked                 15      194        184          0   

                                  Total rows  
road_features                                 
Intersection                            2629  
Intersection with traffic lights        1463  
Intersection with signs                  747  
Intersection unmarked                    233  


For 'Intersection', for all 2629 rows, we can accept the details garnered from the 'road_markings_...' features. 

(i)
For 'Intersection with traffic lights', all 1463 should have:
- road_markings_absent 0
- road_markings_onroad 1
- road_markings_signposts 1 
plus a new category:
- road_markings_traffic_lights 1
unless they already have:
- road_markings_temporary 1

(ii)
For 'Intersection with signs', all 747 should have:
- road_markings_absent 0
- road_markings_signposts 1 
unless they already have:
- road_markings_temporary 1

(iii)
For 'Intersection unmarked', the 233 values 
- road_markings_absent 1
- road_markings_onroad 0
- road_markings_signposts 0 
- road_markings_temporary 0

In [183]:
# (i)  'Intersection with traffic lights'

# Make new column for traffic lights default to 0
if "road_markings_traffic_lights" not in df.columns:
    df["road_markings_traffic_lights"] = 0

# Mask for rows with traffic light intersections, but not temporary markings
mask = (
    (df["road_features"] == "Intersection with traffic lights")
    & (df["road_markings_temporary"] == 0)
)

# Apply changes for these rows
df.loc[mask, "road_markings_absent"] = 0
df.loc[mask, "road_markings_onroad"] = 1
df.loc[mask, "road_markings_signposts"] = 1
df.loc[mask, "road_markings_traffic_lights"] = 1

In [184]:
# (ii) 'Intersection signposted'  NB in Italian this was 'Intersezione stradale segnalata'

# Mask for rows  with Intersection signposted but not temporary markings
mask = (
    (df["road_features"] == "Intersection with signs")
    & (df["road_markings_temporary"] == 0)
)

# Apply changes for these rows
df.loc[mask, "road_markings_absent"] = 0
df.loc[mask, "road_markings_signposts"] = 1
df.loc[mask, "road_markings_onroad"] = 1

In [185]:
# (iii) 'Intersection unmarked'

# Mask for "Intersection unmarked"
mask = df["road_features"] == "Intersection unmarked"

# Apply changes for these rows
df.loc[mask, "road_markings_absent"] = 1
df.loc[mask, "road_markings_onroad"] = 0
df.loc[mask, "road_markings_signposts"] = 0
df.loc[mask, "road_markings_temporary"] = 0

In [186]:
intersection_types = [
    "Intersection",
    "Intersection with traffic lights",
    "Intersection with signs",
    "Intersection unmarked"
]

# Step 1: marking counts table
marking_summary = (
    df[df["road_features"].isin(intersection_types)]
    .groupby("road_features", observed=True)[[
        "road_markings_absent",
        "road_markings_onroad",
        "road_markings_signposts",
        "road_markings_temporary"
    ]]
    .sum()
    .astype(int)
    .reindex(intersection_types)   # keep only these 4, in this order
    .rename(columns={
        "road_markings_absent": "Absent",
        "road_markings_onroad": "On road",
        "road_markings_signposts": "Signposts",
        "road_markings_temporary": "Temporary"
    })
)

# Step 2: true totals per road_feature (count of rows)
true_totals = df["road_features"].value_counts().reindex(
    intersection_types).fillna(0).astype(int)

# Step 3: add as a new column
marking_summary["Total rows"] = true_totals.values

print(marking_summary)

                                  Absent  On road  Signposts  Temporary  \
road_features                                                             
Intersection                         129     2350       2185          5   
Intersection with traffic lights       0     1458       1458          5   
Intersection with signs                0      745        745          2   
Intersection unmarked                233        0          0          0   

                                  Total rows  
road_features                                 
Intersection                            2629  
Intersection with traffic lights        1463  
Intersection with signs                  747  
Intersection unmarked                    233  


In [187]:
# Define mapping
collapse_map = {
    "Intersection unmarked": "Intersection",
    "Intersection with traffic lights": "Intersection",
    "Intersection with signs": "Intersection"
}

# Apply mapping safely, then recast as categorical
df["road_features"] = (
    df["road_features"]
    .astype(str)              # ensure mapping works on all values
    .replace(collapse_map)    # collapse into "Intersection"
)

In [188]:
analyse_column('road_features')

,Count,% per category,Unique Accidents
road_features,,,
Straight road,13009,66.76,12071
Intersection,5072,26.03,4660
Curve with clear view,673,3.45,622
Roundabout,338,1.73,301
Slope,182,0.93,165
Curve without clear view,148,0.76,136
road_feature_other,64,0.33,62


We can also clean up the 'Curve without clear view' rows (which should have 'Visibilita' marked as Insufficiente). Then we can collapse these two categories 'Curve with clear view' and 'Curve without clear view' into one: 'Curve'. 

In [189]:
# Filter only rows where road_features == "Curve without clear view"
subset = df.loc[df["road_features"] ==
                "Curve without clear view", "Visibilita"]

# Quick look at unique values
print("Unique values of Visibilità for 'Curve without clear view':")
print(subset.unique())

# Distribution (counts)
print("\nCounts:")
print(subset.value_counts(dropna=False))

Unique values of Visibilità for 'Curve without clear view':
['Insufficiente' 'Buona' 'Sufficiente']

Counts:
Visibilita
Buona            113
Sufficiente       19
Insufficiente     16
Name: count, dtype: int64


The rows where the 'Visibilita' is 'Buona' or 'Sufficiente' should be changed to 'Insufficiente'

In [190]:
mask = (
    (df["road_features"] == "Curve without clear view")
    & (df["Visibilita"].isin(["Buona", "Sufficiente"]))
)

df.loc[mask, "Visibilita"] = "Insufficiente"

In [191]:
# Define mapping
collapse_map = {
    "Curve with clear view": "Curve",
    "Curve without clear view": "Curve"
}

# Apply mapping safely, then recast as categorical
df["road_features"] = (
    df["road_features"]
    .astype(str)              # ensure mapping works on all values
    .replace(collapse_map)    # collapse into "Curve"
)

# Recreate categorical with only the categories that exist
df["road_features"] = pd.Categorical(df["road_features"])

In [192]:
analyse_column('road_features')

,Count,% per category,Unique Accidents
road_features,,,
Straight road,13009,66.76,12071
Intersection,5072,26.03,4660
Curve,821,4.21,758
Roundabout,338,1.73,301
Slope,182,0.93,165
road_feature_other,64,0.33,62


#### (D) PARTICOLARITASTRADE - road features - CLEANING COMPLETE
NEW CATEGORICAL COLUMNS:
- road_features

### (E) PAVIMENTAZIONE - road surface 

In [193]:
analyse_column('Pavimentazione')

,Count,% per category,Unique Accidents
Pavimentazione,,,
Asfaltata,18433,94.60,17055
In cubetti di porfido,645,3.31,588
Acciotolata,249,1.28,228
Strada pavimentata dissestata,68,0.35,64
Lastricata,51,0.26,44
Con buche,18,0.09,16
Sterrata,11,0.06,11
In conglomerato cementizio,8,0.04,8
Inghiaiata,1,0.01,1


We will group these into four main categories and an 'other' category: Tarmac, Paved, Damaged, Graveled, other. 

In [194]:
# Define mapping
road_surface_map = {
    'Asfaltata': 'Tarmac',
    'In cubetti di porfido': 'Paved',
    'Acciotolata': 'Paved',
    'Strada pavimentata dissestata': 'Damaged',
    'Lastricata': 'Paved',
    'Con buche': 'Damaged',
    'Sterrata': 'Graveled',
    'In conglomerato cementizio': 'Tarmac',
    'Inghiaiata': 'Graveled',
    'Bitumata': 'Tarmac'
}

# Apply mapping and replace all others with 'Unknown'
df['road_surface'] = df['Pavimentazione'].map(
    road_surface_map).fillna('road_surface_unknown')

In [195]:
df['road_surface'].value_counts(dropna=False)

road_surface
Tarmac                  18442
Paved                     945
Damaged                    86
Graveled                   12
road_surface_unknown        1
Name: count, dtype: int64

In [196]:
df.drop(columns='Pavimentazione', inplace=True)

In [197]:
# Recreate categorical with only the categories that exist
df["road_surface"] = pd.Categorical(df["road_surface"])

#### (E) PAVIMENTAZIONE - road surface - CLEANING COMPLETE
CATEGORICAL COLUMNS:
- road_surface

### (F) NATURA INCIDENTE - type of accident

In [198]:
analyse_column('NaturaIncidente')

,Count,% per category,Unique Accidents
NaturaIncidente,,,
Investimento di pedone,19008,97.55,17601
Veicolo in marcia contro veicolo in sosta,97,0.50,82
Veicolo in marcia contro ostacolo accidentale,78,0.40,75
Veicolo in marcia contro veicoli in sosta,76,0.39,61
Infortunio per sola frenata improvvisa,75,0.38,68
Veicolo in marcia contro ostacolo fisso,64,0.33,56
Ribaltamento senza urto contro ostacolo fisso,28,0.14,25
Fuoriuscita dalla sede stradale,15,0.08,11
Veicolo in marcia contro animale,9,0.05,9


We have already eliminated all accidents involving other moving vehicles, therefore in most of these cases, the type of accident is probably incorrect. Police log the type of accident, send statistics off, then double-check and submit the final version of accident details, thus these kinds of errors are common. 

Let us take accidents where a vehicle hits parked vehicle(s) to see how the accidents were categorised:

In [199]:
df.loc[df['parked_vehicle_involved'] == 1, 'NaturaIncidente'].value_counts()

NaturaIncidente
Investimento di pedone                           114
Veicolo in marcia contro veicolo in sosta         48
Veicolo in marcia contro veicoli in sosta         27
Veicolo in marcia contro veicolo fermo             3
Veicolo in marcia contro ostacolo fisso            2
Fuoriuscita dalla sede stradale                    1
Ribaltamento senza urto contro ostacolo fisso      1
Veicolo in marcia contro ostacolo accidentale      1
Name: count, dtype: int64

After examination, we can group these into: 
- accident_pedestrian_run_over
- accident_parked_vehicle_hit
- accident_obstacle_hit
- accident_vehicle_out_of_control

DISCARD (TipoPersona most likely passeggero, not pedone):
- Infortunio per sola frenata improvvisa	(vehicle passengers hurt when stops abruptly)
- Veicolo in marcia contro treno            (car versus train unlikely to involve pedestrian - this outlier can be ignored)

accident_vehicle_out_of_control GROUP (vehicle out of control):
- Ribaltamento senza urto contro ostacolo fisso	
- Fuoriuscita dalla sede stradale	
- Veicolo in marcia contro animale	
- Veicolo in marcia che urta buche nella carreggiata

accident_obstacle_hit GROUP (vehicle hits obstacle):
- Veicolo in marcia contro ostacolo accidentale	
- Veicolo in marcia contro ostacolo fisso	

accident_parked_vehicle_hit GROUP (vehicle hits parked / double-parked car) - ONLY IF PARKED VEHICLES INVOLVED: 
- Veicolo in marcia contro veicolo in sosta	     (IN SOSTA = parked)
- Veicolo in marcia contro veicoli in sosta	
- Veicolo in marcia contro veicolo fermo	     (FERMO = momentarily stopped, ready to continue)
- Veicoli in marcia contro veicoli fermi	

accident_pedestrian_run_over GROUP (likely NaturaIncidente wrong, probably investimento di pedone, given that other vehicles, neither moving nor stationary, are in the accident logs):
IF PARKED VEHICLES NOT INVOLVED
- Veicolo in marcia contro veicolo in sosta	     (IN SOSTA = parked)
- Veicolo in marcia contro veicoli in sosta	
- Veicolo in marcia contro veicolo fermo	     (FERMO = momentarily stopped, ready to continue)
- Veicoli in marcia contro veicoli fermi	
AND
- Investimento di pedone	
- Tamponamento	
- Scontro frontale fra veicoli in marcia	
- Scontro laterale fra veicoli in marcia	
- Scontro frontale/laterale DX fra veicoli in marcia	
- Scontro frontale/laterale SX fra veicoli in marcia	
- Veicolo in marcia contro veicoli in arresto	     (IN ARRESTO = stopped at junction, etc)

Examine 'Infortunio per sola frenata improvvisa': this is when a bus, motorcycle, etc stops suddenly and the passengers fall over / fall off and are hurt. If instead a pedestrian is hit, it is classified 'Investimento di pedone'. 

If the accident involves a bus, motorcycle or scooter (where a passenger can fall), it is likely that the 'pedone' was actually a passenger. 
If the accident involves a car (where a passenger can not fall over/off), it is likely that the accident should be 'Investimento di pedone'. 

In [200]:
# Filter 'infortunio per sola frenata improvvisa' accidents (sudden braking accidents)
mask = df['NaturaIncidente'].eq('Infortunio per sola frenata improvvisa')

# Count by vehicle type
sudden_braking_accidents_by_vehicle_type = (df.loc[mask, 'TipoVeicolo']
                                            .value_counts(dropna=False)
                                            )

print(sudden_braking_accidents_by_vehicle_type)

TipoVeicolo
Motociclo a solo            60
Autovettura privata          5
Autobus urbano               4
Unknown                      2
Autobus di linea             1
Motociclo con passeggero     1
Macchina operatrice          1
Monopattino elettrico        1
Name: count, dtype: int64


Of the sudden braking accidents:
- Those involving private cars (Autovettura privata) were most likely categorised incorrectly and instead should be 'Investimento di pedone' (pedestrian run over). 
- Those involving motorbikes, buses and other vehicles have most likely incorrectly identified the person involved as a pedestrian (pedone), rather than a passenger. These can be dropped.

In [201]:
# Find all rows involving cars
car = df['TipoVeicolo'].eq('Autovettura privata')

# Filter sudden braking accidents where a car was involved, relabel as accident type: pedestrian run over
df.loc[mask & car, 'NaturaIncidente'] = 'Investimento di pedone'

# Drop sudden braking accidents which do not involve cars (bus, moto, etc.)
df = df.loc[~mask | car].reset_index(drop=True)

Double-check there are no more sudden braking accidents:

In [202]:
df['NaturaIncidente'].eq('Infortunio per sola frenata improvvisa').sum()

np.int64(0)

Accidents involving parked vehicles  (IN SOSTA = parked).  
Note that FERMO refers to momentarily stopped (at a stop sign, at a junction or to let someone out, for example). 
Here we take accidents 'contro veicolo/i in sosta' where parked vehicles were involved and change them to 'accident_parked_vehicle_hit'.
Those that did not involved parked vehicles are most likely mistyped and should be 'accident_pedestrian_run_over

In [203]:
mask = (
    df['NaturaIncidente'].isin([
        'Veicolo in marcia contro veicolo in sosta',
        'Veicolo in marcia contro veicoli in sosta'
    ])
    & (df['parked_vehicle_involved'] == 1)
)

df.loc[mask, 'NaturaIncidente'] = 'accident_parked_vehicle_hit'

In [204]:
(df['NaturaIncidente'] == 'accident_parked_vehicle_hit').sum()

np.int64(75)

Previously there were 48 'Veicolo in marcia contro veicolo in sosta' and 27 'Veicolo in marcia contro veicoli in sosta'. 48+27=75. 

The other accident types involving vehicles 'in sosta' were most likely mistyped and should be 'investimento di pedone'. 

In [205]:
accident_type_map = {
    'Investimento di pedone': 'accident_pedestrian_run_over',
    'Tamponamento': 'accident_pedestrian_run_over',
    'Scontro frontale fra veicoli in marcia': 'accident_pedestrian_run_over',
    'Scontro laterale fra veicoli in marcia': 'accident_pedestrian_run_over',
    'Scontro frontale/laterale DX fra veicoli in marcia': 'accident_pedestrian_run_over',
    'Scontro frontale/laterale SX fra veicoli in marcia': 'accident_pedestrian_run_over',
    'Veicolo in marcia contro veicoli in arresto': 'accident_pedestrian_run_over',
    'Veicolo in marcia contro veicolo fermo': 'accident_pedestrian_run_over',
    'Veicoli in marcia contro veicoli fermi': 'accident_pedestrian_run_over',
    'Veicolo in marcia contro veicolo in sosta': 'accident_pedestrian_run_over',
    'Veicolo in marcia contro veicoli in sosta': 'accident_pedestrian_run_over',

    'accident_parked_vehicle_hit': 'accident_parked_vehicle_hit',

    'Veicolo in marcia contro ostacolo accidentale': 'accident_obstacle_hit',
    'Veicolo in marcia contro ostacolo fisso': 'accident_obstacle_hit',

    'Ribaltamento senza urto contro ostacolo fisso': 'accident_vehicle_out_of_control',
    'Fuoriuscita dalla sede stradale': 'accident_vehicle_out_of_control',
    'Veicolo in marcia contro animale': 'accident_vehicle_out_of_control',
    'Veicolo in marcia che urta buche nella carreggiata': 'accident_vehicle_out_of_control'
}

# Apply mapping, default = 'Unknown'
df['accident_type'] = df['NaturaIncidente'].map(
    accident_type_map).fillna('accident_type_unknown')

# Check results
df['accident_type'].value_counts(dropna=False)

accident_type
accident_pedestrian_run_over       19145
accident_obstacle_hit                142
accident_parked_vehicle_hit           75
accident_vehicle_out_of_control       53
accident_type_unknown                  1
Name: count, dtype: int64

In [206]:
# Recreate categorical with only the categories that exist
df["accident_type"] = pd.Categorical(df["accident_type"])

Now we can drop the 'NaturaIncidente' column, as well as the columns refering to the number of injured, dead, etc ('NUM_FERITI','NUM_RISERVATA', 'NUM_MORTI', 'NUM_ILLESI') as this information is already available in these columns:
- 'Tipolesione' 
- 'Deceduto' 
- 'DecedutoDopo',
- 'driver_injury'
- 'driver_deceduto'
- 'driver_deceduto_dopo'
- 'passengers_killed'
- 'passengers_uninjured'
- 'passengers_injured'

In [207]:
df.drop(columns=['NaturaIncidente', 'NUM_FERITI',
        'NUM_RISERVATA', 'NUM_MORTI', 'NUM_ILLESI'], inplace=True)

In [208]:
df.columns

Index(['Protocollo', 'TipoVeicolo', 'StatoVeicolo', 'DataOraIncidente',
       'CondizioneAtmosferica', 'Traffico', 'Gruppo', 'Localizzazione1',
       'STRADA1', 'Localizzazione2', 'STRADA2', 'Strada02', 'Chilometrica',
       'DaSpecificare', 'Latitude', 'Longitude', 'TipoStrada', 'Visibilita',
       'Illuminazione', 'Tipolesione', 'Deceduto', 'DecedutoDopo',
       'parked_vehicle_involved', 'driver_injury', 'driver_deceduto',
       'driver_deceduto_dopo', 'driver_gender', 'passenger_no_of_females',
       'passenger_no_of_males', 'passenger_no_of_unknown_sex',
       'passengers_killed', 'passengers_uninjured', 'passengers_injured',
       'phase_of_day', 'YEAR', 'MONTH', 'DATE', 'TIME', 'DAY',
       'pedestrian_gender', 'road_conditions', 'road_markings_absent',
       'road_markings_onroad', 'road_markings_signposts',
       'road_markings_temporary', 'road_features',
       'road_markings_traffic_lights', 'road_surface', 'accident_type'],
      dtype='object')

In [209]:
df.to_parquet('007_data_only_pedestrians.parquet', index=False)

In [210]:
df.shape

(19416, 49)

In [ ]:
metadata = {
    'notebook': '007 Data cleaning encoding.ipynb',
    'step': 'categorical encoding, feature collapsing, and schema hygiene',

    # IO
    'initial_parquet_file': '006_data_only_pedestrians.parquet',
    'final_parquet_file': '007_data_only_pedestrians.parquet',

    # Shapes
    'initial_rows': 19_687,
    'initial_columns': 49,
    'intermediate_shapes': [
        {'name': 'after_drop_missing_8of10', 'shape': (19_686, 49)},
        {'name': 'after_drop_missing_9of10', 'shape': (19_486, 49)}
    ],
    'final_rows': 19_416,
    'final_columns': 49,

    # Row removals (detail)
    # among {Sesso, Traffico, Localizzazione1, particolaritastrade, TipoStrada, FondoStradale, Pavimentazione, Segnaletica, Visibilita, Illuminazione}
    'rows_removed_missing_8of10': 1,
    'rows_removed_missing_9of10': 200,
    # dropped "Infortunio per sola frenata improvvisa" unless TipoVeicolo == 'Autovettura privata'
    'rows_removed_sudden_braking_non_car': 70,
    'rows_removed_total': 271,            # 19,687 → 19,416

    # Columns
    'columns_added': [
        'road_markings_absent',
        'road_markings_onroad',
        'road_markings_signposts',
        'road_markings_temporary',
        'road_surface',        # from Pavimentazione
        'road_features',       # from particolaritastrade (collapsed)
        'road_conditions',     # from FondoStradale
        # from NaturaIncidente (+ parked_vehicle_involved rule)
        'accident_type'
    ],
    'columns_removed': [
        'Sesso',
        'FondoStradale',
        'Segnaletica',
        'particolaritastrade',
        'Pavimentazione',
        'NaturaIncidente',
        'NUM_FERITI',
        'NUM_RISERVATA',
        'NUM_MORTI',
        'NUM_ILLESI'
    ],
    'columns_modified': [
        "Visibilita: for road_features == 'Curve without clear view', {'Buona','Sufficiente'} → 'Insufficiente'"
    ],
    'categorical_casts': [
        'driver_gender', 'pedestrian_gender',
        'road_conditions', 'road_surface', 'road_features', 'accident_type'
    ],

    # Encodings / mappings applied (high level)
    'encodings_applied': {
        'road_markings_from_Segnaletica': {
            'produced': ['road_markings_absent', 'road_markings_onroad', 'road_markings_signposts', 'road_markings_temporary']
        },
        'road_surface_from_Pavimentazione': ['Tarmac', 'Paved', 'Damaged', 'Graveled', 'road_surface_unknown'],
        'road_conditions_from_FondoStradale': ['dry', 'wet', 'slippery', 'icy', 'road_conditions_unknown'],
        'road_features_from_particolaritastrade': 'mapped → collapsed intersections → {Straight road, Intersection, Curve, Roundabout, Slope, road_feature_other}',
        'accident_type_from_NaturaIncidente': {
            'rules': [
                "If parked_vehicle_involved==1 and NaturaIncidente in {'...veicolo in sosta...'} → 'accident_parked_vehicle_hit'",
                "Recode sudden-braking with car to 'Investimento di pedone', drop sudden-braking without car",
                "Map to classes: {accident_pedestrian_run_over, accident_obstacle_hit, accident_parked_vehicle_hit, accident_vehicle_out_of_control}; else 'accident_type_unknown'"
            ]
        }
    },

    # QA snapshots (from value_counts printed in notebook)
    'qa_value_counts': {
        'road_markings': {
            'absent': 1_366,
            'onroad': 17_291,
            'signposts': 15_098,
            'temporary': 57
        },
        'road_conditions': {     # snapshot before final 70-row drop
            'dry': 16_179, 'wet': 3_276, 'slippery': 17, 'icy': 13, 'unknown': 1
        },
        'road_surface': {        # snapshot before final 70-row drop
            'Tarmac': 18_442, 'Paved': 945, 'Damaged': 86, 'Graveled': 12, 'road_surface_unknown': 1
        },
        'road_features': {       # snapshot before final 70-row drop (collapsed)
            'Straight road': 13_009, 'Intersection': 5_072, 'Curve': 821, 'Roundabout': 338, 'Slope': 182, 'road_feature_other': 64
        },
        'accident_type': {       # final snapshot (sums to 19,416)
            'accident_pedestrian_run_over': 19_145,
            'accident_obstacle_hit': 142,
            'accident_parked_vehicle_hit': 75,
            'accident_vehicle_out_of_control': 53,
            'accident_type_unknown': 1
        }
    },

    # Decisions (narrative)
    'decisions_made': [
        'Dropped rows with extreme missingness across 10 key roadway/visibility columns (8/10 and 9/10 missing).',
        "Encoded Segnaletica into four binary indicators and removed the original column.",
        "Standardised Pavimentazione → road_surface; FondoStradale → road_conditions; particolaritastrade → road_features (with intersection categories collapsed).",
        "Normalised Visibilita for 'Curve without clear view' to Insufficiente where it was Buona/Sufficiente.",
        "Recoded NaturaIncidente using parked_vehicle_involved; consolidated into high-level accident_type classes.",
        "Removed raw source and unused count columns once the engineered features were created."
    ]
}